In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [3]:
#Load Data

In [4]:
df=pd.read_csv('loans_full_schema.csv')
df_raw=df.copy()
df.head()

,Unnamed: 0,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,1,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,2,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,3,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,4,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,5,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0


In [5]:
df.shape

(10000, 56)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 56 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Unnamed: 0                        10000 non-null  int64  
 1   emp_title                         9167 non-null   object 
 2   emp_length                        9183 non-null   float64
 3   state                             10000 non-null  object 
 4   homeownership                     10000 non-null  object 
 5   annual_income                     10000 non-null  float64
 6   verified_income                   10000 non-null  object 
 7   debt_to_income                    9976 non-null   float64
 8   annual_income_joint               1495 non-null   float64
 9   verification_income_joint         1455 non-null   object 
 10  debt_to_income_joint              1495 non-null   float64
 11  delinq_2y                         10000 non-null  int64  
 12  month

In [8]:
df['emp_length']=df['emp_length'].fillna(df['emp_length'].mean())

In [7]:
#Dealing With Missing Data

In [9]:
df['debt_to_income'].isnull().sum()

np.int64(24)

In [10]:
((df['annual_income']==0 )& (df['debt_to_income'].isnull())).sum()

np.int64(23)

In [11]:
((df['annual_income']!=0 )& (df['debt_to_income'].isnull())).sum()

np.int64(1)

In [12]:
df['annual_income'][(df['annual_income']!=0 )& (df['debt_to_income'].isnull())]

2106    1.0
Name: annual_income, dtype: float64

In [13]:
df=df.drop(2106)

In [14]:
(df['debt_to_income'].isnull() & df['annual_income_joint'].isnull()).sum()

np.int64(0)

whenever 'debt to income ' is null then 'annual income ' is zero except in one case that is on data position '2106' so we have remove that exception so that it do nor create issue in further analysis and 'annual joint income ' is not zero or null.this means that 'debt to income 'is null becuse there is 0 annual income,but data can't be dropped because there is joint account in those position as there is no null for 'annual joint income ' when 'debt to income' is null.Althogh this does not imply that there is joint account only when annual income is 0 as there are around 1471 position where "annual joint income' is non null which indicates that in those position there are joint account.
Also we analysed that whenever "debt_to_income" is null "joint annual income " is not null which indicates that we can take "joint annual income" feauture as strong indicator of account to be joint or not.we will try to look if it is 100% indicator for such or not

In [15]:
#df.loc[(df['debt_to_income'].isnull()) & (df['debt_to_income_joint'].notnull()), ['debt_to_income','debt_to_income_joint','annual_income','annual_income_joint']].info()

In [16]:
#df.loc[~(df['debt_to_income'].isnull()) & (df['debt_to_income_joint'].notnull()),['debt_to_income','debt_to_income_joint','annual_income','annual_income_joint']].head()

In [17]:
#dealing with missing debt_to_equity values

In [18]:
df['flag_debt_to_income_null']=df['debt_to_income'].notnull().astype(int)

In [19]:
df['debt_to_income'] = df['debt_to_income'].fillna(df['debt_to_income_joint'])

In [20]:
#Dealing with missing title

In [21]:
df['emp_title']=df['emp_title'].fillna("Unknown")

In [22]:
#dealing with missing annual income joint

In [23]:
df['annual_income_joint'].notnull().sum()

np.int64(1494)

In [24]:
df['annual_income_joint'].isnull().sum()

np.int64(8505)

In [25]:
(df['annual_income_joint']==0).sum()

np.int64(0)

so basically there are 8505 null values and 1494 non null value which mean there are at least 1494 joint account

In [26]:
df[['annual_income','debt_to_income','annual_income_joint','debt_to_income_joint']][(df['annual_income_joint'].notnull()) & (df['annual_income']==0)].info()

<class 'pandas.core.frame.DataFrame'>
Index: 23 entries, 154 to 9273
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   annual_income         23 non-null     float64
 1   debt_to_income        23 non-null     float64
 2   annual_income_joint   23 non-null     float64
 3   debt_to_income_joint  23 non-null     float64
dtypes: float64(4)
memory usage: 920.0 bytes


In [27]:
df[['annual_income','debt_to_income','annual_income_joint','debt_to_income_joint']] .loc[(df['annual_income_joint'].notnull()) & (df['annual_income']!=0)].info()

<class 'pandas.core.frame.DataFrame'>
Index: 1471 entries, 4 to 9997
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   annual_income         1471 non-null   float64
 1   debt_to_income        1471 non-null   float64
 2   annual_income_joint   1471 non-null   float64
 3   debt_to_income_joint  1471 non-null   float64
dtypes: float64(4)
memory usage: 57.5 KB


So basically there are 23  position where ;annual joint income' is not null and and 'annual income' is zero .also we got to know that these 23 position are same for which "debt to equity" was null because"annual income was 0 on those position
there are 1471 position where both 'joint annual income' and 'annual income'are not zero.this indicates that there are 1471 people who have both 'joint income and solo income'
At this point we can surely say that whenever "annual_income_joint " is only 100% indicator of joint account


In [28]:
df['annual_income_joint'].isnull().sum()

np.int64(8505)

In [29]:
#dealing with missing value of 'Joint_annual_income'

Before dealing with missing value in "Annual_income_joint" we will use it as indicator wheather joint account or not

In [30]:
df['has_joint_applicant'] = df['annual_income_joint'].notnull().astype(int)

In [31]:
df['annual_income_joint'] = df['annual_income_joint'].fillna(0)

In [32]:
df['annual_income_joint'].info()

<class 'pandas.core.series.Series'>
Index: 9999 entries, 0 to 9999
Series name: annual_income_joint
Non-Null Count  Dtype  
--------------  -----  
9999 non-null   float64
dtypes: float64(1)
memory usage: 156.2 KB


In [33]:
(df['has_joint_applicant']==0).sum()

np.int64(8505)

In [34]:
#Dealing with missing 

In [35]:
df['verification_income_joint']=df['verification_income_joint'].fillna("Missing")

In [36]:
df['verification_income_joint'].info()

<class 'pandas.core.series.Series'>
Index: 9999 entries, 0 to 9999
Series name: verification_income_joint
Non-Null Count  Dtype 
--------------  ----- 
9999 non-null   object
dtypes: object(1)
memory usage: 156.2+ KB


In [37]:
df[['annual_income_joint','annual_income']][(df['debt_to_income_joint'].isnull()) & (df['has_joint_applicant']==0)].info()

<class 'pandas.core.frame.DataFrame'>
Index: 8505 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   annual_income_joint  8505 non-null   float64
 1   annual_income        8505 non-null   float64
dtypes: float64(2)
memory usage: 199.3 KB


we analysed that there are 8505 position where "debt to income joint " is null and at those position there is no joint apllication which indicates that 
null values in "debt to income joint" is because there is missing or zero joint income

In [38]:
df['debt_to_income_joint']=df['debt_to_income_joint'].fillna(df['debt_to_income'])

here we dont need to map  joint account as we have earlier we have mapped all the joint account by using "annual income joint" feauture

In [39]:
#Dealing with missing values in 'months_since_last_delinq'

In [40]:
df['Never_delinqued']=df['months_since_last_delinq'].notnull().astype(int)

In [41]:
df['months_since_last_delinq'].info()

<class 'pandas.core.series.Series'>
Index: 9999 entries, 0 to 9999
Series name: months_since_last_delinq
Non-Null Count  Dtype  
--------------  -----  
4342 non-null   float64
dtypes: float64(1)
memory usage: 156.2 KB


In [59]:
df['months_since_last_delinq'].fillna(120)

0        38.0
1       120.0
2        28.0
3       120.0
4       120.0
        ...  
9995    120.0
9996      9.0
9997      6.0
9998    120.0
9999    120.0
Name: months_since_last_delinq, Length: 9999, dtype: float64

Using reasonable cap to fill the missing value assuming that missing values is because borrower never delequent

In [43]:
df.isnull().sum()

Unnamed: 0                             0
emp_title                              0
emp_length                             0
state                                  0
homeownership                          0
annual_income                          0
verified_income                        0
debt_to_income                         0
annual_income_joint                    0
verification_income_joint              0
debt_to_income_joint                   0
delinq_2y                              0
months_since_last_delinq               0
earliest_credit_line                   0
inquiries_last_12m                     0
total_credit_lines                     0
open_credit_lines                      0
total_credit_limit                     0
total_credit_utilized                  0
num_collections_last_12m               0
num_historical_failed_to_pay           0
months_since_90d_late               7714
current_accounts_delinq                0
total_collection_amount_ever           0
current_installm

In [44]:
#Dealing with missing values of "months_since_90d_late"

In [45]:
df['months_since_90d_late'].info()

<class 'pandas.core.series.Series'>
Index: 9999 entries, 0 to 9999
Series name: months_since_90d_late
Non-Null Count  Dtype  
--------------  -----  
2285 non-null   float64
dtypes: float64(1)
memory usage: 156.2 KB


In [46]:
df['Never_been-90days_late']=df['months_since_90d_late'].notnull().astype(int)

In [47]:
df['months_since_90d_late'].max()

128.0

In [48]:
df['months_since_90d_late']=df['months_since_90d_late'].fillna(129)

In [49]:
df['Never_been-90days_late'][(df['Never_been-90days_late']==0)].info()

<class 'pandas.core.series.Series'>
Index: 7714 entries, 1 to 9999
Series name: Never_been-90days_late
Non-Null Count  Dtype
--------------  -----
7714 non-null   int64
dtypes: int64(1)
memory usage: 120.5 KB


Filling missing values of 'months_since_90d_late' by filling missing values with max+1 assuming that borrower at missing value never 
been 90 days late

In [50]:
df['months_since_last_credit_inquiry'].isnull().sum()

np.int64(1271)

In [51]:
#Dealing with missing values in 'months_since_last_credit_inquiry' 

In [52]:
df['Never_have_credit_inquiry']=df['months_since_last_credit_inquiry'].notnull().astype(int)

In [53]:
df['months_since_last_credit_inquiry']=df['months_since_last_credit_inquiry'].fillna(999)

filling missing values in 'months_since_last_credit_inquiry' with high value 999 because this feauture is not strongly linked with default unlike "months_since_90d_late' which was strngly linked with default

In [54]:
#Dealing with missing values in 'num_accounts_120d_past_due' 

In [55]:
df['num_accounts_120d_past_due'][(df['num_accounts_120d_past_due']!=0)]

7      NaN
18     NaN
30     NaN
50     NaN
66     NaN
        ..
9900   NaN
9905   NaN
9921   NaN
9968   NaN
9972   NaN
Name: num_accounts_120d_past_due, Length: 318, dtype: float64

In [58]:
df['missing_120d_past_due'] = df['num_accounts_120d_past_due'].isna().astype(int)
df['num_accounts_120d_past_due'].fillna(0)

0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
9995    0.0
9996    0.0
9997    0.0
9998    0.0
9999    0.0
Name: num_accounts_120d_past_due, Length: 9999, dtype: float64

here is missing value does not mean that there is 0 account which us 120 days past dues as ther are entries as 0 to represent that.Here missing values most probably indicating that there is not enogh information.so we have to do mapping for such accounts for which there is no entry about number of account with 120 days due for better analysis of data


In [57]:
df.isnull().sum()

Unnamed: 0                   0
emp_title                    0
emp_length                   0
state                        0
homeownership                0
                            ..
has_joint_applicant          0
Never_delinqued              0
Never_been-90days_late       0
Never_have_credit_inquiry    0
missing_120d_past_due        0
Length: 62, dtype: int64